# R_M19 -- Variance decomposition ICC

Model M19 (Variance decomposition ICC) in R, on the panel R_P00 prepared, with the design set in THIS notebook (v20.57: applied when it runs). Every result carries the design-based SE (the honest benchmark) and is flagged if its SE is ~0.

In [ ]:
# the bundle's R folder (works from rstudio/, jupyter/ or the project root)
R_HOME_DIR <- normalizePath(if (file.exists("lib/reward_paths.R")) "." else "..", winslash = "/")
for (f in c("reward_paths.R", "reward_design.R", "reward_prep.R", "reward_models_core.R")) source(file.path(R_HOME_DIR, "lib", f), local = environment())
cat("data root:", ROOT, "| output:", OUTPUT_DIR, "\n")
invisible(confirm_packages())   # v20.55: are ALL R packages installed? (pre-built first; a missing one is installed now when AUTO_INSTALL_PACKAGES)

**Settings** -- the outcome (`ALL_OUTCOMES <- TRUE` runs every outcome) and THE DESIGN OF THIS RUN (v20.57): timing (your fund workbook by default), dose, control rings, years, seasons, fragments, overlaps, fixed effects, covariates -- applied when this model runs, on the panel R_P00 built once. Change them here and re-run this notebook only.

In [ ]:
OUTCOME      <- "NDVI"
ALL_OUTCOMES <- FALSE
# ---- v20.57: THE DESIGN OF THIS RUN -- every option below is applied HERE, when this model runs, on the panel R_P00 built ONCE.
#      Change any of them and re-run this notebook: R_P00 is never re-run for it. What you set is what runs -- DESIGN IN EFFECT
#      (printed by model_design(), saved as DESIGN_IN_EFFECT.csv next to the results) lists each option, the value used and why.
DESIGN_MODE      <- "recommended"   # "recommended": an option set to "data" is chosen FROM THE DATA (DESIGN_RECOMMENDATION.md: export breaks,
                                    #   fill years, spillover into inner rings -- never the effect) | "manual": "data" = every ring / every year.
                                    #   A value you set (e.g. CONTROL_RINGS <- 1:3, PRE_YEARS <- 4) is ALWAYS used exactly as set, in both modes
TREATMENT_TIMING <- "fund"          # "fund": each sub-watershed's first treated SEASON from your fund workbook -- the release timing, back-cast
                                    #   before the file's first month, the season AFTER the start (no anticipation): staggered, season-level
                                    #   cohorts (results/FUND/FUND_TIMING_AND_DOSE) | "registry": its year in data/sites/sites.csv | "fixed"
TREATMENT_YEAR   <- 2022            # "fixed" timing -- and the fall-back for a sub-watershed the fund file / registry does not date (announced)
FUND_START_RULE  <- "backcast"      # fund timing: "backcast" | "share" (the amount reaches 10 % of the target) | "file_start" (a lower bound)
DOSE_VARIABLE    <- "dose_intensity_per_ha"   # the dose (fund file): amount released / treatment area | "dose_amount_sws" | "dose_share_of_target"
                                              #   -- 0 on the control rings and before the treatment starts
CONTROL_RINGS    <- "data"          # control ring(s): "data" | 1:5 | 1:3 | 2:5 | c(1, 3, 5)
PRE_YEARS        <- "data"          # years BEFORE the start: "data" | NA (every year) | 4 (the 4 years before)
POST_YEARS       <- "data"          # years FROM the start:   "data" | NA (every year) | 2 (the start year + 1)
SEASONS          <- "all"           # "all" (annual composite + Kharif/Rabi/Zaid: year AND season variation) | "seasonal" | "yearly" |
                                    #   "Rabi" | c("Kharif", "Rabi") | "auto" (the data decide)
EXCLUDE_TRANSITION_YEAR <- FALSE    # TRUE = the first treated year of each series leaves the sample (robustness)
UNIT_FE          <- "pixel_season"  # one fixed effect per pixel x season series (+ year x season effects) | "pixel"
SUB_WATERSHEDS   <- "data"          # v20.58: the sub-watershed(s) THIS run processes: "data" = every one with >= 5 % of the largest one's rows (one
                                    #   export = its own sub-watershed) | "major" (the largest only) | a name or id, e.g. "Koranahalli" | c(1, 7)
FRAGMENT_RULE    <- "drop"          # YOUR RULE: rows of sub-watersheds NOT processed and rows outside every polygon leave EVERY group -- treated and
                                    #   control, pre and post (SAMPLE INTEGRITY confirms it at every run) | "keep"
OVERLAP_ROWS     <- "drop"          # YOUR RULE: overlapping pixels leave every group -- a pixel treated in another processed sub-watershed (as a
                                    #   control), a repeated pixel-year-season, a near-duplicate pixel, a pixel whose ring the exports disagree on | "keep"
POOLED_FE        <- "site_period"   # several sub-watersheds: sub-watershed x year x season effects | "period" (one shared year x season effect)
EXCLUDE_GAPFILLED <- TRUE           # rows the exporter filled from history (GapFilled / Coverage 0) are not observations | FALSE keeps them
COVARIATES       <- c("Rain", "Tmax", "Tmean", "Tmin")   # the weather covariates of this model | character(0) = none


**Run** -- load the screened panel, estimate, save (results under ROOT/output/results).

In [ ]:
design <- model_design()      # v20.57: the settings above -> the design of THIS run (DESIGN IN EFFECT)
res <- run_model_R("M19", OUTCOME, design)
if (ALL_OUTCOMES) for (o in setdiff(OUTCOMES, OUTCOME)) run_model_R("M19", o, design)